# 🏛️ Adaptación de Sonata a la Catedral (auto-supervisada, **sin etiquetas**)

Continúa el preentrenamiento de Sonata (el mismo objetivo tipo DINO del paper) con **todos los escaneos de la Catedral menos `PL7` y `PL13`**, que se dejan fuera para evaluar. Después extrae el nuevo `f_ssl` de `PL7` y `PL13` y lo guarda en tu Drive.

**Estructura del Drive** (la celda 1 la crea y acomoda los archivos viejos sin pisar nada):

```
PFC1_HyperNCD_Sonata/
├── m1/entrada, m1/salida        data_m1.tar.gz va en m1/salida
├── m2/entrada, m2/salida        checkpoints de HyperNCD y resultados de M2
├── m3/entrenamiento             checkpoints (model_last.pth) y train.log
├── m3/pesos                     sonata_catedral.pth (pesos adaptados)
├── m3/salida                    PL7_..._feats.npz, PL13_..._feats.npz (f_ssl adaptado)
├── m4/salida                    grupos y métricas de la fusión
└── entornos/                    pointcept_env_<tag>.tar.gz, hyperncd_env_cache.tar.gz
```

**Qué necesita:** `m1/salida/data_m1.tar.gz` (la carpeta `data/` de Khipu con los `.npy` del Módulo 1).

**Entorno compilado:** la primera vez tarda ~15-25 min; las siguientes lo restaura de `entornos/` en ~1-2 min. Si se cae la sesión, el entrenamiento se reanuda desde `m3/entrenamiento`.

**Orden:** corre las celdas 1 → 8 en orden. Si reconectas, vuelve a correr 1, 2, 3 y 4 (rápidas por la caché) y sigue desde donde quedaste.

> ⚠️ Este cuaderno se escribió sin poder probarlo en Colab. Si una celda falla, copia el error completo y se corrige. Las celdas 4 y 6 tienen comprobaciones que paran con un mensaje claro si algo no cuadra.

In [ ]:
# @title 1. GPU, Google Drive y estructura de carpetas por módulo
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, torch
from google.colab import drive
drive.mount('/content/drive')

# ---- Estructura del Drive: una carpeta por módulo (crea carpetas y acomoda los archivos viejos) ----
import os, shutil, glob

DRIVE = "/content/drive/MyDrive/PFC1_HyperNCD_Sonata"
M1_ENT, M1_SAL = f"{DRIVE}/m1/entrada", f"{DRIVE}/m1/salida"
M2_ENT, M2_SAL = f"{DRIVE}/m2/entrada", f"{DRIVE}/m2/salida"
M3_ENTRENA, M3_PESOS, M3_SAL = f"{DRIVE}/m3/entrenamiento", f"{DRIVE}/m3/pesos", f"{DRIVE}/m3/salida"
M4_SAL   = f"{DRIVE}/m4/salida"
ENTORNOS = f"{DRIVE}/entornos"
for _d in (M1_ENT, M1_SAL, M2_ENT, M2_SAL, M3_ENTRENA, M3_PESOS, M3_SAL, M4_SAL, ENTORNOS):
    os.makedirs(_d, exist_ok=True)

def _mover(origen, destino):
    """Mueve origen -> destino solo si origen existe. Nunca pisa nada: si destino existe, no lo toca."""
    if not os.path.exists(origen):
        return
    if os.path.exists(destino):
        if os.path.isdir(origen) and os.path.isdir(destino):   # fusiona carpetas
            for x in os.listdir(origen):
                _mover(os.path.join(origen, x), os.path.join(destino, x))
            try:
                os.rmdir(origen)
            except OSError:
                pass
        else:
            print("  ya existe, no se toca:", destino.replace(DRIVE + "/", ""))
        return
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    shutil.move(origen, destino)
    print("  movido:", origen.replace(DRIVE + "/", ""), "->", destino.replace(DRIVE + "/", ""))

# M1: la salida (los .npy preprocesados, empaquetados)
_mover(f"{DRIVE}/data_m1.tar.gz",    f"{M1_SAL}/data_m1.tar.gz")
_mover(f"{DRIVE}/m1/data_m1.tar.gz", f"{M1_SAL}/data_m1.tar.gz")
# M2: checkpoints de entrada y resultados
_mover(f"{DRIVE}/datos_m2_descomprimidos",         f"{M2_ENT}/datos_m2_descomprimidos")
_mover(f"{DRIVE}/resultados_m2",                   f"{M2_SAL}/resultados_m2")
_mover(f"{DRIVE}/resultados_modulo2_hyperncd.zip", f"{M2_SAL}/resultados_modulo2_hyperncd.zip")
# Entornos compilados
_mover(f"{DRIVE}/hyperncd_env_cache.tar.gz", f"{ENTORNOS}/hyperncd_env_cache.tar.gz")
for _p in glob.glob(f"{DRIVE}/pointcept_env_*.tar.gz"):
    _mover(_p, f"{ENTORNOS}/{os.path.basename(_p)}")
# M3: si ya existía la carpeta anterior de la adaptación
_mover(f"{DRIVE}/sonata_adaptado/entrenamiento",       M3_ENTRENA)
_mover(f"{DRIVE}/sonata_adaptado/sonata_catedral.pth", f"{M3_PESOS}/sonata_catedral.pth")
_mover(f"{DRIVE}/sonata_adaptado/features",            M3_SAL)
print("Estructura lista en", DRIVE)

print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))

In [ ]:
# @title 2. Datos: descomprimir data_m1.tar.gz y armar el formato de entrenamiento
import os, glob, shutil, subprocess
import numpy as np

TAR      = f"{M1_SAL}/data_m1.tar.gz"         # salida de M1 (entrada de M2 y M3)
EXCLUIR  = ["PL7_SCN0001", "PL13_SCN0001"]    # @param {type:"raw"}  escaneos que NO se usan para entrenar
DATA_RAW = "/content/data_m1"
DATA_PC  = "/content/pc_data/catedral"        # carpeta que lee Pointcept (train/<escena>/coord.npy ...)

assert os.path.exists(TAR), f"Falta {TAR}. Sube data_m1.tar.gz a la carpeta m1/salida de tu Drive."
if not os.path.isdir(f"{DATA_RAW}/data"):
    os.makedirs(DATA_RAW, exist_ok=True)
    subprocess.run(["tar", "-xzf", TAR, "-C", DATA_RAW], check=True)

npys = sorted(glob.glob(f"{DATA_RAW}/data/*.npy"))
print(f"{len(npys)} escaneos encontrados")

shutil.rmtree(DATA_PC, ignore_errors=True)
usados = []
for ruta in npys:
    nombre = os.path.basename(ruta)[:-4]
    if nombre in EXCLUIR:
        print(f"  (fuera, para evaluar) {nombre}")
        continue
    d = np.load(ruta, mmap_mode="r")
    assert d.ndim == 2 and d.shape[1] == 9, f"{nombre}: se esperaban 9 columnas, hay {d.shape}"
    out = f"{DATA_PC}/train/{nombre}"
    os.makedirs(out, exist_ok=True)
    np.save(f"{out}/coord.npy",  np.ascontiguousarray(d[:, 0:3], dtype=np.float32))
    np.save(f"{out}/color.npy",  np.ascontiguousarray(d[:, 3:6], dtype=np.float32))
    np.save(f"{out}/normal.npy", np.ascontiguousarray(d[:, 6:9], dtype=np.float32))
    cmax = float(np.max(d[:, 3:6]))
    usados.append(nombre)
    print(f"  {nombre:28s} {d.shape[0]:>10,d} puntos | color max {cmax:.0f}")
    if cmax <= 1.5:
        print("   ⚠️ color parece estar en 0-1; Sonata espera 0-255 (igual que en extraer_features.py)")
print(f"\nEntrenarán {len(usados)} escaneos.")

In [ ]:
# @title 3. Entorno de Pointcept (con caché en Drive: la 1ª vez compila, las siguientes restauran)
import os, sys, subprocess, torch

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if check and r.returncode != 0:
        raise RuntimeError(f"Falló (código {r.returncode}): {cmd}")

PYV  = f"py{sys.version_info.major}{sys.version_info.minor}"
TV   = torch.__version__.split("+")[0]
CU   = (torch.version.cuda or "cpu").replace(".", "")
TAG  = f"{PYV}_torch{TV}_cu{CU}"
ENV_TAR = f"{ENTORNOS}/pointcept_env_{TAG}.tar.gz"
VENV = "/content/pcenv"
PYBIN = f"{VENV}/bin/python"
PIP   = f"{VENV}/bin/pip"
POINTCEPT = "/content/Pointcept"
CAP = "%d.%d" % torch.cuda.get_device_capability(0)

if not os.path.isdir(POINTCEPT):
    sh(f"git clone --depth 1 https://github.com/Pointcept/Pointcept {POINTCEPT}")

if os.path.exists(ENV_TAR) and not os.path.exists(PYBIN):
    print(f"⚡ Restaurando entorno desde Drive: {ENV_TAR}")
    sh(f'tar -xzf "{ENV_TAR}" -C /content')
elif not os.path.exists(PYBIN):
    print(f"🔧 Entorno {TAG} no está en Drive. Construyéndolo (15-25 min la primera vez)...")
    sh("pip install -q virtualenv")
    # --system-site-packages: reutiliza el torch de Colab (no se reinstala)
    sh(f"virtualenv -q --system-site-packages {VENV}")
    sh(f"{PIP} install -q setuptools wheel ninja")
    sh(f"{PIP} install -q addict yapf einops h5py plyfile termcolor timm tensorboardX tensorboard "
       f"wandb SharedArray open3d ftfy regex tqdm huggingface_hub scipy pyyaml packaging")
    if CU.startswith("11"):
        spc = "spconv-cu118"
    elif CU.startswith("12"):
        spc = "spconv-cu120"
    else:
        spc = "spconv-cu126"   # no hay ruedas de spconv para CUDA 13 (Colab nuevo); es un intento
        print(f"⚠️ CUDA {torch.version.cuda} y Python {sys.version.split()[0]} son más nuevos que lo que soporta spconv. "
              "Puede fallar; la comprobación de abajo dirá qué módulo.")
    sh(f"{PIP} install -q {spc}")
    # torch-scatter: rueda precompilada de PyG si existe; si no, se compila
    url = f"https://data.pyg.org/whl/torch-{TV}+cu{CU}.html"
    r = subprocess.run(f"{PIP} install -q torch-scatter --only-binary=:all: -f {url}", shell=True, executable="/bin/bash")
    if r.returncode != 0:
        print("No hay rueda de torch-scatter para esta versión de torch; compilando (~10 min)...")
        sh(f"TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation torch-scatter")
    # pointops (extensión CUDA de Pointcept)
    sh(f"cd {POINTCEPT}/libs/pointops && TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation .")
    # el resto de dependencias que importa pointcept.models / pointcept.datasets
    sh(f"{PIP} install -q peft transformers torch-geometric")
    # torch-cluster: no hay rueda para torch tan nuevo; se compila (~5-10 min)
    sh(f"TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation torch-cluster")
    _recien_construido = True

# pointgroup_ops (segmentacion de instancias) no se usa en el preentrenamiento de Sonata: modulo vacio
os.makedirs("/content/stubs", exist_ok=True)
open("/content/stubs/pointgroup_ops.py", "w").write("# stub: no se usa en Sonata")

# --- Comprobación módulo por módulo (así se ve CUÁL falla y por qué) ---
ENV = dict(os.environ, PYTHONPATH=f"{POINTCEPT}:/content/stubs")
MODULOS = ["torch", "spconv.pytorch", "torch_scatter", "pointops", "timm", "addict", "wandb",
           "tensorboardX", "open3d", "torch_cluster", "torch_geometric", "peft", "transformers", "pointcept.models", "pointcept.datasets"]
fallos = {}
for m in MODULOS:
    r = subprocess.run([PYBIN, "-c", f"import {m}"], cwd=POINTCEPT, env=ENV, capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  ✅ {m}")
    else:
        ultima = (r.stderr.strip().splitlines() or ["(sin mensaje)"])[-1]
        fallos[m] = ultima
        print(f"  ❌ {m}: {ultima}")

if fallos:
    print("\nENTORNO INCOMPLETO. Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
    print("No se guardó en Drive. Si ya había una copia mala en entornos/, bórrala:", ENV_TAR)
    raise RuntimeError("Faltan módulos: " + ", ".join(fallos))

if globals().get("_recien_construido"):
    print("💾 Entorno OK. Guardando en Drive para la próxima vez...")
    sh(f'tar -czf "{ENV_TAR}" -C /content pcenv')
print("✅ Entorno listo.")

In [ ]:
# @title 4. Configuración de entrenamiento + pesos iniciales de Sonata (se verifica que carguen)
import os, re, torch, subprocess

EPOCHS     = 30        # @param {type:"integer"}  épocas (cada época recorre las escenas LOOP veces)
LOOP       = 5         # @param {type:"integer"}  repeticiones de cada escena por época (hay pocas escenas)
BATCH      = 1         # @param {type:"integer"}  escenas por lote (1 para T4 de 15 GB; 2-4 con 24-40 GB)
MAX_PUNTOS = 32768     # @param {type:"integer"}  puntos máx. por vista (original 65536; baja a 16384 si sigue "out of memory")
LR         = 0.0002    # @param {type:"number"}   lr base (el original, 0.004, es para entrenar desde cero)
NUM_WORKER = 2         # @param {type:"integer"}

SAVE   = M3_ENTRENA
os.makedirs(SAVE, exist_ok=True)
CFG_REL = "configs/sonata/pretrain-sonata-catedral.py"

# --- Parche de memoria: atención con SDPA de PyTorch (funciona en T4) en vez de la matriz K x K de Pointcept ---
# Sin flash-attn, Pointcept guarda una matriz (N', H, 1024, 1024) por bloque y se queda sin VRAM.
_f = f"{POINTCEPT}/pointcept/models/point_transformer_v3/point_transformer_v3m2_sonata.py"
_src = open(_f, encoding="utf-8").read()
_viejo = """            attn = (q * self.scale) @ k.transpose(-2, -1)  # (N', H, K, K)
            if self.enable_rpe:
                attn = attn + self.rpe(self.get_rel_pos(point, order))
            if self.upcast_softmax:
                attn = attn.float()
            attn = self.softmax(attn)
            attn = self.attn_drop(attn).to(qkv.dtype)
            feat = (attn @ v).transpose(1, 2).reshape(-1, C)
"""
_orig_indentado = "".join(("    " + l) if l.strip() else l for l in _viejo.splitlines(True))
_nuevo = """            if (not self.enable_rpe) and (not self.upcast_softmax):
                feat = torch.nn.functional.scaled_dot_product_attention(
                    q, k, v.to(q.dtype),
                    dropout_p=self.attn_drop.p if self.training else 0.0,
                    scale=self.scale,
                )
                feat = feat.transpose(1, 2).reshape(-1, C).to(qkv.dtype)
            else:
""" + _orig_indentado
if "scaled_dot_product_attention" in _src:
    print("Parche de atención ya aplicado.")
else:
    assert _viejo in _src, "No encontré el bloque de atención de Pointcept (cambió el repo?)"
    open(_f, "w", encoding="utf-8").write(_src.replace(_viejo, _nuevo))
    print("✅ Parche de atención (SDPA) aplicado.")

base = open(f"{POINTCEPT}/configs/sonata/pretrain-sonata-v1m1-0-base.py", encoding="utf-8").read()
i, j = base.index("data = dict("), base.index("hooks = [")
data_block = """data = dict(
    train=dict(
        type="DefaultDataset",
        split="train",
        data_root="/content/pc_data/catedral",
        transform=transform,
        test_mode=False,
        loop=__LOOP__,
    )
)

""".replace("__LOOP__", str(LOOP))
cfg = base[:i] + data_block + base[j:]

AMP = "bfloat16" if torch.cuda.get_device_capability(0)[0] >= 8 else "float16"
cambios = [
    ("batch_size = 96",            f"batch_size = {BATCH}"),
    ("num_worker = 96",            f"num_worker = {NUM_WORKER}"),
    # Pointcept fija loop = epoch // eval_epoch (pisa el loop del dataset). Para que cada época haga LOOP pasadas
    # y se entrene EPOCHS épocas: epoch = EPOCHS*LOOP, eval_epoch = EPOCHS.
    ("epoch = 200",                f"epoch = {EPOCHS * LOOP}\neval_epoch = {EPOCHS}"),
    ("base_lr = 0.004",            f"base_lr = {LR}"),
    ("enable_flash=True",          "enable_flash=False"),
    ('amp_dtype = "bfloat16"',     f'amp_dtype = "{AMP}"'),
    ("save_freq=5",                "save_freq=5"),
    ("max_size=65536",             f"max_size={MAX_PUNTOS}"),
]
for viejo, nuevo in cambios:
    assert viejo in cfg, f"No encontré '{viejo}' en la config base de Pointcept (cambió el repo?)"
    cfg = cfg.replace(viejo, nuevo)
cfg += "\n# sin wandb (pide clave de API); el registro queda en train.log y tensorboard\nenable_wandb = False\n"
open(f"{POINTCEPT}/{CFG_REL}", "w", encoding="utf-8").write(cfg)
print(f"Config escrita: {CFG_REL}  (epochs={EPOCHS}, loop={LOOP}, batch={BATCH}, lr={LR}, amp={AMP})")

# --- Pesos iniciales: sonata.pth -> claves de student.backbone y teacher.backbone ---
from huggingface_hub import hf_hub_download
p_sonata = hf_hub_download("facebook/sonata", "sonata.pth")
ck = torch.load(p_sonata, map_location="cpu", weights_only=True)
nuevo = {}
for k, v in ck["state_dict"].items():
    nuevo[f"student.backbone.{k}"] = v
    nuevo[f"teacher.backbone.{k}"] = v
INIT = "/content/sonata_init.pth"
torch.save({"state_dict": nuevo}, INIT)
torch.save(ck, "/content/sonata_original_ckpt.pth")

# --- Verificación: ¿esas claves existen en el modelo de preentrenamiento de Pointcept? ---
chk = """
import sys, torch
from pointcept.utils.config import Config
from pointcept.models import build_model
cfg = Config.fromfile(sys.argv[1])
m = build_model(cfg.model)
claves_modelo = set(m.state_dict().keys())
init = torch.load(sys.argv[2], map_location="cpu", weights_only=False)["state_dict"]
ok = [k for k in init if k in claves_modelo]
mal = [k for k in init if k not in claves_modelo]
print(f"claves iniciales: {len(init)} | coinciden con el modelo: {len(ok)} | no coinciden: {len(mal)}")
print("ejemplos que no coinciden:", mal[:5])
sys.exit(0 if len(ok) >= 0.9 * len(init) else 3)
"""
open("/content/chk_keys.py", "w").write(chk)
r = subprocess.run([PYBIN, "/content/chk_keys.py", f"{POINTCEPT}/{CFG_REL}", INIT], cwd=POINTCEPT, env=ENV)
assert r.returncode == 0, "Las claves de sonata.pth no coinciden con el modelo de Pointcept. Copia la salida de arriba."
print("✅ Pesos iniciales compatibles.")

In [ ]:
# @title 5. ENTRENAR (guarda en Drive; si se cae la sesión, vuelve a correr 1-4 y esta celda con REANUDAR=True)
REANUDAR = False   # @param {type:"boolean"}

import os, subprocess
ultimo = f"{SAVE}/model/model_last.pth"
if REANUDAR and os.path.exists(ultimo):
    opciones = f"save_path={SAVE} weight={ultimo} resume=True"
    print("↻ Reanudando desde", ultimo)
else:
    opciones = f"save_path={SAVE} weight={INIT} resume=False"
    print("▶ Entrenamiento nuevo (parte de Sonata preentrenado)")

cmd = (f'cd {POINTCEPT} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=./:/content/stubs {PYBIN} -u tools/train.py '
       f'--config-file {CFG_REL} --num-gpus 1 --options {opciones} 2>&1 | tee -a "{SAVE}/train.log"')
subprocess.run(cmd, shell=True, executable="/bin/bash")
print("\nCheckpoints en:", SAVE + "/model")
print("Para ver la VRAM mientras entrena, corre !nvidia-smi en otra celda (o mira memory/mem en el log).")

In [ ]:
# @title 6. Exportar pesos adaptados al formato de `sonata.load` (usa el teacher)
import torch, os

ORIG = torch.load("/content/sonata_original_ckpt.pth", map_location="cpu", weights_only=True)
FINAL = f"{SAVE}/model/model_last.pth"
assert os.path.exists(FINAL), f"No hay {FINAL}: ¿terminó al menos una época?"
ck = torch.load(FINAL, map_location="cpu", weights_only=False)["state_dict"]

adaptado, mantenidos, sin_forma = {}, 0, 0
for k, v0 in ORIG["state_dict"].items():
    clave = None
    for pref in ("teacher.backbone.", "module.teacher.backbone."):
        if pref + k in ck:
            clave = pref + k
            break
    if clave is not None and ck[clave].shape == v0.shape:
        adaptado[k] = ck[clave].clone()
    else:
        adaptado[k] = v0
        mantenidos += 1
n = len(ORIG["state_dict"])
print(f"{n - mantenidos}/{n} tensores vienen del entrenamiento; {mantenidos} se dejaron del Sonata original")
assert n - mantenidos > 0.9 * n, "Casi nada coincide: revisa nombres de claves en el checkpoint de Pointcept"

# ¿cuánto cambiaron los pesos respecto al original? (0 = nada se entrenó)
import math
num = sum(((adaptado[k].float() - ORIG["state_dict"][k].float()) ** 2).sum().item() for k in adaptado)
den = sum((ORIG["state_dict"][k].float() ** 2).sum().item() for k in adaptado)
print(f"Cambio relativo de pesos: {math.sqrt(num / den):.4%}")

PESOS = f"{M3_PESOS}/sonata_catedral.pth"
torch.save({"config": ORIG["config"], "state_dict": adaptado}, PESOS)
print("💾", PESOS)

In [ ]:
# @title 7. Clonar tu repo (rama main) para usar extraer_features.py
import os
from getpass import getpass

REPO_OWNER, REPO_NAME, BRANCH = "manyorycuevamendoza", "PFC1-HyperNCD-Sonata", "main"
REPO_DIR = f"/content/{REPO_NAME}"

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        print("✅ Token leído desde Colab Secrets (GITHUB_TOKEN).")
except Exception:
    pass
if not token:
    token = getpass("Ingresa tu GitHub Personal Access Token (oculto): ").strip()

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
%cd {REPO_DIR}
!git log -1 --oneline
assert "--pesos" in open(f"{REPO_DIR}/extraer_features.py", encoding="utf-8").read(), \
    "Tu rama no tiene la opción --pesos en extraer_features.py: haz push de ese cambio primero."

In [ ]:
# @title 8. Extraer f_ssl adaptado de PL7 y PL13 → Drive
import os, subprocess

ESCANEOS = ["PL7_SCN0001", "PL13_SCN0001"]    # @param {type:"raw"}
OUT_FEATS = M3_SAL
os.makedirs(OUT_FEATS, exist_ok=True)

ENV2 = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/sonata")
for nombre in ESCANEOS:
    entrada = f"{DATA_RAW}/data/{nombre}.npy"
    salida  = f"{OUT_FEATS}/{nombre}_adaptado_feats.npz"
    print(f"\n=== {nombre} ===")
    r = subprocess.run([PYBIN, "extraer_features.py", "--input", entrada, "--output", salida,
                        "--pesos", PESOS], cwd=REPO_DIR, env=ENV2,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-6000:])          # lo que imprimió el script (incluye el error)
    assert r.returncode == 0, f"Falló la extracción de {nombre} (mira el error de arriba)"
print("\n✅ Listo. Descarga de Drive:", OUT_FEATS)
!ls -lh "{OUT_FEATS}"

## Después, en tu PC
1. Baja `features/PL13_SCN0001_adaptado_feats.npz` a `experimentacion/datos/modulo3_PL13/`.
2. Corre `experimentacion/modulo4_real/correr_modulo4_real.py` apuntando `--fssl` a ese archivo y compara con el `f_ssl` original (ARI, estabilidad, y mIoU cuando esté el recorte etiquetado).
3. Compara contra el Sonata original con las mismas etiquetas: esa es la comparación que importa.